In [1]:
# Imports and paths 

from pathlib import Path

from pypdf import PdfReader
from langchain_core.documents import Document

RAW_DIR = Path("../data/raw")
print(sorted(RAW_DIR.glob("*.pdf")))

[WindowsPath('../data/raw/conditions_generales_bienvenue_2026.pdf'), WindowsPath('../data/raw/tarifs_particuliers_2026.pdf')]


In [2]:
# Look at what pypdf extracts

pdf_path = RAW_DIR / "tarifs_particuliers_2026.pdf"
reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))
print("=" * 80)
print(reader.pages[3].extract_text())

Number of pages: 36
04
OUVERTURE, FONCTIONNEMENT 
ET SUIVI DE VOTRE COMPTE
  Ouverture, transformation, clôture
•  Ouverture de compte Gratuit
•  Transfert de compte dans une autre agence BNP Paribas Gratuit
•  Clôture de compte Gratuit
•   Service d’aide à la mobilité bancaire Facilimouv’  Gratuit 
(conforme aux articles L.312-1-7 et R.312-4-4 du Code monétaire et financier)  
Ce service permet un changement automatisé de la domiciliation vers votre nouveau  
compte de l’ensemble des prélèvements et virements récurrents enregistrés sur  
un compte ouvert dans une autre banque au cours des 13 derniers mois.
•   Service de changement de domiciliation bancaire Facilidom Gratuit 
Vous choisissez les émetteurs à informer de vos nouvelles coordonnées bancaires.  
Arrêt de la commercialisation le 01/01/2026 et suppression définitive du service Facilidom le 30/04/2026.
  Relevés de compte
•   Relevé standard (en ligne ou papier)  
- Mensuel Gratuit 
- Autre périodicité : décadaire, quotidien 

In [3]:
# Our own loader 

def load_pdf(path: Path) -> list[Document]:
    reader = PdfReader(path)
    documents = []
    for page_number, page in enumerate(reader.pages, start=1):
        documents.append(
            Document(
                page_content=page.extract_text(),
                metadata={"source": path.name, "page": page_number},
            )
        )
    return documents

In [4]:
# Load both PDF

documents = []
for pdf_path in sorted(RAW_DIR.glob("*.pdf")):
    pages = load_pdf(pdf_path)
    print(f"{pdf_path.name}: {len(pages)} pages")
    documents.extend(pages)

print("Total documents:", len(documents))

conditions_generales_bienvenue_2026.pdf: 20 pages
tarifs_particuliers_2026.pdf: 36 pages
Total documents: 56


In [5]:
# Inspect one document 

first = documents[0]
print(first.metadata)
print("=" * 80)
print(first.page_content[:1000])

{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 1}
1
CONVENTION DE COMPTE DE DÉPÔT  
BIENVENUE 
CONDITIONS GÉNÉRALES
SOMMAIRE
ÉDITION AVRIL 2026
TITRE I –  LES MODALITÉS DE SOUSCRIPTION  
DE LA CONVENTION P 2
TITRE II – LA TENUE DU COMPTE DE DÉPÔT P 3
I  - LES CONDITIONS D’OUVERTURE DU COMPTE P 3 
II  - LE FONCTIONNEMENT DU COMPTE  P 3
III  - LES MOYENS DE PAIEMENT  P 5   
IV  - LA PROCURATION P 10 
V  - LA FACILITÉ DE CAISSE AUTOMATIQUE  P 10 
VI  -  L’INDISPONIBILITÉ DES ACTIFS PAR SUITE DE SAISIE   P 11 
VII  - LE TRANSFERT DU COMPTE   P 11 
VIII  -  LE SORT DU COMPTE EN CAS DE DÉCÈS P 11 
IX  - LES CONDITIONS TARIFAIRES  P 12
X  - LE SERVICE CASCADE P 12
TITRE III – DISPOSITIONS DIVERSES  P 12
I  - CHAMP D’APPLICATION DE LA CONVENTION  P 12
II  - OBLIGATIONS DÉCLARATIVES DE LA BANQUE P 13
III  - DURÉE ET RÉSILIATION DE LA CONVENTION  
- CLÔTURE DU COMPTE  P 13
IV  - MODIFICATIONS DE LA CONVENTION  P 14
V  - RÉSOUDRE UN LITIGE  P 14
VI  - GARANTIE DES DÉPÔTS  P 15
VII 

In [6]:
# Find nearly empty pages 

for doc in documents:
    n_chars = len(doc.page_content)
    if n_chars < 300:
        print(doc.metadata, n_chars, repr(doc.page_content[:80]))

{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 18} 299 '18\nII - AUTRES ANNEXES\nLes autres annexes de la Convention sont :\n  ●  « Conditi'
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 19} 10 '19\nNOTES\xa0:'
{'source': 'tarifs_particuliers_2026.pdf', 'page': 1} 108 'CONDITIONS  \nET TARIFS\nProduits et services  \npour les particuliers \nTarifs appl'


In [7]:
# First, very simple search engine 

def find(keyword: str) -> None:
    for doc in documents:
        if keyword.lower() in doc.page_content.lower():
            print(doc.metadata)


find("opposition")
print("-" * 40)
find("on m'a volé ma carte")

{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 3}
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 6}
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 9}
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 11}
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 15}
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 16}
{'source': 'tarifs_particuliers_2026.pdf', 'page': 8}
{'source': 'tarifs_particuliers_2026.pdf', 'page': 10}
{'source': 'tarifs_particuliers_2026.pdf', 'page': 18}
{'source': 'tarifs_particuliers_2026.pdf', 'page': 20}
{'source': 'tarifs_particuliers_2026.pdf', 'page': 35}
----------------------------------------


In [8]:
# Display a full page 

def show_page(source: str, page: int) -> None:
    for doc in documents:
        if doc.metadata["source"] == source and doc.metadata["page"] == page:
            print(doc.page_content)


show_page("conditions_generales_bienvenue_2026.pdf", 1)
print("=" * 80)
show_page("tarifs_particuliers_2026.pdf", 2)

1
CONVENTION DE COMPTE DE DÉPÔT  
BIENVENUE 
CONDITIONS GÉNÉRALES
SOMMAIRE
ÉDITION AVRIL 2026
TITRE I –  LES MODALITÉS DE SOUSCRIPTION  
DE LA CONVENTION P 2
TITRE II – LA TENUE DU COMPTE DE DÉPÔT P 3
I  - LES CONDITIONS D’OUVERTURE DU COMPTE P 3 
II  - LE FONCTIONNEMENT DU COMPTE  P 3
III  - LES MOYENS DE PAIEMENT  P 5   
IV  - LA PROCURATION P 10 
V  - LA FACILITÉ DE CAISSE AUTOMATIQUE  P 10 
VI  -  L’INDISPONIBILITÉ DES ACTIFS PAR SUITE DE SAISIE   P 11 
VII  - LE TRANSFERT DU COMPTE   P 11 
VIII  -  LE SORT DU COMPTE EN CAS DE DÉCÈS P 11 
IX  - LES CONDITIONS TARIFAIRES  P 12
X  - LE SERVICE CASCADE P 12
TITRE III – DISPOSITIONS DIVERSES  P 12
I  - CHAMP D’APPLICATION DE LA CONVENTION  P 12
II  - OBLIGATIONS DÉCLARATIVES DE LA BANQUE P 13
III  - DURÉE ET RÉSILIATION DE LA CONVENTION  
- CLÔTURE DU COMPTE  P 13
IV  - MODIFICATIONS DE LA CONVENTION  P 14
V  - RÉSOUDRE UN LITIGE  P 14
VI  - GARANTIE DES DÉPÔTS  P 15
VII  - DONNÉES PERSONNELLES  P 15
VIII  - SECRET BANCAIRE  P 15 
IX  

In [9]:
cg_page_1 = documents[0].page_content
print(len(cg_page_1.splitlines()), "lines in total")
print("=" * 80)
print(cg_page_1[-700:])

35 lines in total
 -  LE SORT DU COMPTE EN CAS DE DÉCÈS P 11 
IX  - LES CONDITIONS TARIFAIRES  P 12
X  - LE SERVICE CASCADE P 12
TITRE III – DISPOSITIONS DIVERSES  P 12
I  - CHAMP D’APPLICATION DE LA CONVENTION  P 12
II  - OBLIGATIONS DÉCLARATIVES DE LA BANQUE P 13
III  - DURÉE ET RÉSILIATION DE LA CONVENTION  
- CLÔTURE DU COMPTE  P 13
IV  - MODIFICATIONS DE LA CONVENTION  P 14
V  - RÉSOUDRE UN LITIGE  P 14
VI  - GARANTIE DES DÉPÔTS  P 15
VII  - DONNÉES PERSONNELLES  P 15
VIII  - SECRET BANCAIRE  P 15 
IX  - INFORMATION RELATIVE AU DÉMARCHAGE  
   TÉLÉPHONIQUE  P 16
X  - LOI APPLICABLE, TRIBUNAUX COMPÉTENTS, LANGUE P 16
XI  - DÉCLARATIONS ET ENGAGEMENTS DU CLIENT  P 16
XII  - ACCESSIBILITÉ P 16
ANNEXES P 17



In [10]:
import sys

sys.path.append("..")

from src.loader import load_all_documents

documents = load_all_documents()
print("Total documents:", len(documents))
print(documents[0].metadata)

Total documents: 53
{'source': 'conditions_generales_bienvenue_2026.pdf', 'page': 2, 'title': 'Conditions générales de la convention de compte Bienvenue', 'version': 'Édition avril 2026'}


In [1]:
import re
import sys

sys.path.append("..")

from src.loader import load_all_documents

documents = load_all_documents()
card_pages = [doc for doc in documents if doc.metadata["doc_type"] == "card_contract"]
print("Total pages:", len(documents))
print("Card contract pages:", len(card_pages))
print()

for doc in card_pages:
    text = doc.page_content
    match = re.search(r"perte|\bvol\b", text, flags=re.IGNORECASE)
    if match:
        start = max(0, match.start() - 100)
        snippet = text[start : match.start() + 200].replace("\n", " ")
        print(f"p.{doc.metadata['page']}: ...{snippet}...")
        print()

Total pages: 69
Card contract pages: 16

p.7: ...egistrements comme justification de leur imputation au Compte. 10.2 BNP Paribas est responsable des pertes directes encourues par le Titulaire de la Carte dues à une déficience technique du système de  paiement sur lequel BNP Paribas a un contrôle direct. Toutefois, BNP Paribas n’est pas tenu pour r...

p.8: ...de blocage) Ces dispositions ne concernent pas la Carte Virtuelle. Les opérations consécutives à la perte ou au vol de la Carte sont à la charge du Titulaire de la Carte dans la limite de 50 euros. Toutefois,   sa responsabilité n’est pas engagée :   ● en cas d’opération de paiement effectuée sans u...

p.9: ... :   ● du montant de l’opération contestée de bonne foi par le Titulaire de la Carte dans le cas de perte et/ou vol, d’utilisation frauduleuse et/ou de  détournement de sa Carte et des données qui y sont liées, survenue avant la demande d’opposition (ou de blocage) conformément à l’article  12.2  ; ...



In [2]:
for doc in card_pages:
    if re.search(r"^\s*12\.", doc.page_content, flags=re.MULTILINE):
        print(f"=============== p.{doc.metadata['page']} ===============")
        print(doc.page_content)
        print()

=============== p.7 ===============
7
SMONCAROSCGPDF0301
  ● un Accepteur donneur d’ordre ayant signé un contrat d’acceptation de vente à distance sécurisé avec un acquéreur adhérant au schéma de 
Cartes de paiement dont la(les) marques figure(nt) sur la Carte.
La mise en œuvre du service de réception de fonds est soumise aux conditions suivantes :
  ● BNP Paribas a contrôlé au préalable l’éligibilité de la Carte au service ;
  ● BNP Paribas a répondu favorablement à une demande d’autorisation en crédit de l’émetteur du donneur d’ordre ;
  ● BNP Paribas procède au crédit du Compte dans le délai de maximum d’un jour à réception des fonds de l’opération de transfert de fonds.
Lorsque BNP Paribas reçoit une demande d’autorisation aux fins de crédit du Compte du Titulaire bénéficiaire, il est en mesure de rejeter 
l’opération en cas d’impossibilité de réaliser l’opération de transfert de fonds notamment en cas de Compte fermé ou de carte de paiement en 
opposition.
Ce service n’est actuell

In [3]:
for doc in card_pages:
    match = re.search(r"^\s*12\.1", doc.page_content, flags=re.MULTILINE)
    if match:
        start = max(0, match.start() - 300)
        print(f"=============== p.{doc.metadata['page']} ===============")
        print(doc.page_content[start : match.start() + 2500])

=============== p.7 ===============
ée sans délai, par tout moyen écrit, sur support papier ou durable.
En cas de contestation de cette demande d’opposition (ou de blocage), celle-ci sera réputée avoir été effectuée à la date de la réception dudit 
écrit par BNP Paribas.
XII – RESPONSABILITÉ DU TITULAIRE DE LA CARTE ET DE BNP PARIBAS
12.1 Principe
Le Titulaire de la Carte doit prendre toute mesure pour conserver sa Carte et préserver les Données de Sécurité Personnalisées qui lui sont 
attachées, notamment son Code. Il doit l’utiliser conformément aux finalités spécifiées à l’article I (Partie 1 du Contrat).
Le Titulaire de la Carte assume, comme indiqué à l’article 12.2, les conséquences de l’utilisation de la Carte tant qu’il n’a pas fait une demande 
d’opposition (ou de blocage) dans les conditions prévues à l’article XI.
Lorsque le Titulaire de la Carte est mineur non émancipé, le Représentant légal est tenu solidairement des éventuelles conséquences 
dommageables pouvant découler 

In [4]:
page_7 = next(doc for doc in card_pages if doc.metadata["page"] == 7)
text = page_7.page_content

start = re.search(r"^\s*XI\s*[–-]", text, flags=re.MULTILINE).start()
end = re.search(r"^\s*XII\s*[–-]", text, flags=re.MULTILINE).start()

print(text[start:end])

XI – RECEVABILITÉ DES DEMANDES D’OPPOSITION OU DE BLOCAGE
Pour l’exécution du Contrat, l’information ci-dessus visée « de blocage » peut également être désignée par le terme « d’opposition ».
11.1 Dès qu’il a connaissance de la perte ou du vol de la Carte, de son détournement ou de toute utilisation frauduleuse de la Carte ou des 
données liées à son utilisation, le Titulaire de la Carte et/ou du Compte doit en informer sans tarder BNP Paribas aux fins de blocage de sa Carte 
en indiquant les motifs pour lesquels il demande le blocage.
11.2 Cette demande d’opposition (ou de blocage) doit être faite :
  ● auprès de BNP Paribas en agence par déclaration écrite remise sur place, pendant les heures d’ouverture, accessibles sur le site Internet 
mabanque.bnpparibas (1), ou sur l’application mobile « Mes Comptes » (1) ou auprès des Conseillers Banque en Ligne notamment par téléphone, 
ou par Internet dans l’espace client mabanque.bnpparibas (1) ;
  ● ou d’une façon générale auprès du Centre 

In [5]:
for doc in card_pages:
    n_chars = len(doc.page_content)
    if n_chars < 300:
        print(doc.metadata["page"], n_chars, repr(doc.page_content[:80]))